# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [10]:
# This cell is for CODE (numbers, a query, a check).

import duckdb
import pandas as pd
from google.colab import userdata

MONTH = "2026-03"          # same mid-panel month as Week 3, for continuity
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_TABLE = f"{REL}/fact_content_daily_performance/month={MONTH}/*.parquet"

con = duckdb.connect()
token = userdata.get("HF_TOKEN")
con.sql(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

# Same content-level aggregation as Week 3, extended with a raw session count —
# we need real volume (not just rates) to test whether engagement survives a floor.
q_signals = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0)             AS ctr,
    SUM(ga4_engaged_sessions)::DOUBLE / NULLIF(SUM(ga4_sessions), 0)      AS engagement_rate,
    SUM(ga4_total_engagement_sec)::DOUBLE / NULLIF(SUM(ga4_sessions), 0)  AS session_depth_sec,
    AVG((gsc_data_available IS TRUE)::INT)                                AS gsc_availability_rate,
    SUM(ga4_sessions)                                                     AS sessions_month
FROM read_parquet('{FACT_TABLE}')
GROUP BY client_hash_id, content_hash_id
"""
sig = con.sql(q_signals).df()
sig_valid = sig.dropna(subset=["engagement_rate", "gsc_availability_rate"]).copy()
print(f"Total rows: {len(sig)}  |  Rows with computable rates: {len(sig_valid)}")
print("gsc_availability_rate distribution:")
print(sig_valid["gsc_availability_rate"].describe())
print()
print(sig["gsc_availability_rate"].value_counts(bins=10, dropna=False).sort_index())
print()

print("What's actually driving the row loss?")
print(f"sig total rows:                 {len(sig)}")
print(f"null gsc_availability_rate:      {sig['gsc_availability_rate'].isna().sum()}")
print(f"null engagement_rate:            {sig['engagement_rate'].isna().sum()}")
print(f"null both:                       {(sig['gsc_availability_rate'].isna() & sig['engagement_rate'].isna()).sum()}")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 331437  |  Rows with computable rates: 90237
gsc_availability_rate distribution:
count    90237.000000
mean         0.640055
std          0.418460
min          0.000000
25%          0.100000
50%          0.903226
75%          1.000000
max          1.000000
Name: gsc_availability_rate, dtype: float64

(-0.002, 0.1]    178775
(0.1, 0.2]        11906
(0.2, 0.3]         7960
(0.3, 0.4]         6560
(0.4, 0.5]         7405
(0.5, 0.6]         6553
(0.6, 0.7]         8804
(0.7, 0.8]         9123
(0.8, 0.9]        10456
(0.9, 1.0]        83895
Name: count, dtype: int64

What's actually driving the row loss?
sig total rows:                 331437
null gsc_availability_rate:      0
null engagement_rate:            241200
null both:                       0


In [11]:
# This cell is for CODE (numbers, a query, a check).

# Does the panel-fill flag actually change the numbers, or just the row count?
q_corrected = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(CASE WHEN ga4_data_available THEN ga4_engaged_sessions END)::DOUBLE
        / NULLIF(SUM(CASE WHEN ga4_data_available THEN ga4_sessions END), 0)   AS engagement_rate_filtered,
    SUM(CASE WHEN ga4_data_available THEN ga4_sessions END)                    AS sessions_filtered
FROM read_parquet('{FACT_TABLE}')
GROUP BY client_hash_id, content_hash_id
"""
corrected = con.sql(q_corrected).df()

compare = sig.merge(corrected, on=["client_hash_id", "content_hash_id"])
print("Null engagement_rate — naive (all days) vs flag-filtered (available days only):")
print(f"  naive:     {compare['engagement_rate'].isna().sum()}")
print(f"  filtered:  {compare['engagement_rate_filtered'].isna().sum()}")

both_valid = compare.dropna(subset=["engagement_rate", "engagement_rate_filtered"])
diff = (both_valid["engagement_rate"] - both_valid["engagement_rate_filtered"]).abs()
print(f"\nRows with a value both ways: {len(both_valid)}")
print(f"Median absolute change from filtering: {diff.median():.4f}")
print(f"Rows changed by more than 0.05: {(diff > 0.05).sum()}  ({(diff > 0.05).mean():.1%})")

q_diag = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(ga4_engaged_sessions)::DOUBLE / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,
    SUM(ga4_sessions)                                                AS sessions_month,
    SUM(gsc_impressions)                                             AS impressions_month,
    MAX(ga4_data_available)::INT                                     AS ever_ga4_available
FROM read_parquet('{FACT_TABLE}')
GROUP BY client_hash_id, content_hash_id
"""
diag = con.sql(q_diag).df()

print("Null engagement_rate, split by whether GA4 was ever available this month for that page:")
split = diag.assign(is_null=diag["engagement_rate"].isna()).groupby("ever_ga4_available")["is_null"]
print(split.agg(n_null="sum", n_total="count", null_rate="mean"))

q_diag2 = f"""
WITH per_row AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN ga4_data_available IS TRUE  THEN 'available'
            WHEN ga4_data_available IS FALSE THEN 'not_yet_onboarded'
            ELSE 'unknown_null'
        END AS ga4_status
    FROM read_parquet('{FACT_TABLE}')
),
per_content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        MAX((ga4_status = 'available')::INT)         AS has_any_available,
        MAX((ga4_status = 'not_yet_onboarded')::INT) AS has_any_not_onboarded,
        MAX((ga4_status = 'unknown_null')::INT)      AS has_any_unknown
    FROM per_row
    GROUP BY client_hash_id, content_hash_id
)
SELECT has_any_available, has_any_not_onboarded, has_any_unknown, COUNT(*) AS n_content_pairs
FROM per_content
GROUP BY 1, 2, 3
ORDER BY n_content_pairs DESC
"""
con.sql(q_diag2).show()

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Null engagement_rate — naive (all days) vs flag-filtered (available days only):
  naive:     241200
  filtered:  241200

Rows with a value both ways: 90237
Median absolute change from filtering: 0.0000
Rows changed by more than 0.05: 0  (0.0%)
Null engagement_rate, split by whether GA4 was ever available this month for that page:
                    n_null  n_total  null_rate
ever_ga4_available                            
0                   170248   170248   1.000000
1                      252    90489   0.002785
┌───────────────────┬───────────────────────┬─────────────────┬─────────────────┐
│ has_any_available │ has_any_not_onboarded │ has_any_unknown │ n_content_pairs │
│       int32       │         int32         │      int32      │      int64      │
├───────────────────┼───────────────────────┼─────────────────┼─────────────────┤
│                 0 │                     1 │               0 │          111981 │
│                 0 │                     0 │               1 │       

In [12]:
# This cell is for CODE (numbers, a query, a check).

# ---------- Signal 1: low visibility, still engaging? ----------
sig_valid["visibility_tier"] = pd.cut(
    sig_valid["gsc_availability_rate"],
    bins=[-0.001, 0.10, 0.90, 1.0],
    labels=["low", "mid", "high"],
)
print(sig_valid["visibility_tier"].value_counts())
signal1_table = sig_valid.groupby("visibility_tier", observed=True).agg(
    n=("content_hash_id", "count"),
    median_ctr=("ctr", "median"),
    median_engagement_rate=("engagement_rate", "median"),
    median_session_depth_sec=("session_depth_sec", "median"),
)
print("\n--- Signal 1: visibility tier vs engagement ---")
print(signal1_table)

# ---------- Signal 2 (flag-linked, behind is_quick_win): does volume change the story? ----------
sig_valid["volume_tier"] = pd.cut(
    sig_valid["sessions_month"],
    bins=[0, 1, 9, sig_valid["sessions_month"].max()],
    labels=["1_session", "2_to_9", "10_plus"],
)
print(sig_valid["volume_tier"].value_counts())

signal2_table = sig_valid.groupby("volume_tier", observed=True).agg(
    n=("content_hash_id", "count"),
    median_engagement_rate=("engagement_rate", "median"),
    pct_extreme_engagement=("engagement_rate", lambda s: ((s <= 0.01) | (s >= 0.99)).mean()),
    median_sessions=("sessions_month", "median"),
)
print("\n--- Signal 2: volume tier vs engagement reliability ---")
print(signal2_table)

# Where do your candidate hidden gems actually live, by volume?
candidates = sig_valid[
    (sig_valid["visibility_tier"] == "low")
    & (sig_valid["engagement_rate"] >= sig_valid["engagement_rate"].median())
]
print(f"\nHidden-gem candidates (n={len(candidates)}) by volume tier:")
print(candidates["volume_tier"].value_counts())

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

visibility_tier
high    45600
low     22602
mid     22035
Name: count, dtype: int64

--- Signal 1: visibility tier vs engagement ---
                     n  median_ctr  median_engagement_rate  \
visibility_tier                                              
low              22602    0.000000                     0.0   
mid              22035    0.000000                     0.0   
high             45600    0.001838                     0.0   

                 median_session_depth_sec  
visibility_tier                            
low                                   0.0  
mid                                   0.0  
high                                  0.0  
volume_tier
2_to_9       35925
1_session    33105
10_plus      21207
Name: count, dtype: int64

--- Signal 2: volume tier vs engagement reliability ---
                 n  median_engagement_rate  pct_extreme_engagement  \
volume_tier                                                          
1_session    33105                     0.0  

In [13]:
# This cell is for CODE (numbers, a query, a check).

sig_valid["engaged_sessions_month"] = sig_valid["engagement_rate"] * sig_valid["sessions_month"]

signal1_weighted = sig_valid.groupby("visibility_tier", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "median_engagement_rate": g["engagement_rate"].median(),      # keep the old one for comparison
        "weighted_engagement_rate": g["engaged_sessions_month"].sum() / g["sessions_month"].sum(),
        "median_sessions": g["sessions_month"].median(),
    })
)
print(signal1_weighted)

pd.set_option("display.precision", 10)

raw = sig_valid.groupby("visibility_tier", observed=True).agg(
    n=("content_hash_id", "count"),
    sum_engaged=("engaged_sessions_month", "sum"),
    sum_sessions=("sessions_month", "sum"),
)
raw["weighted_rate"] = raw["sum_engaged"] / raw["sum_sessions"]
print(raw)

pd.reset_option("display.precision")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

                       n  median_engagement_rate  weighted_engagement_rate  \
visibility_tier                                                              
low              22602.0                     0.0                  0.022265   
mid              22035.0                     0.0                  0.027650   
high             45600.0                     0.0                  0.022265   

                 median_sessions  
visibility_tier                   
low                          1.0  
mid                          2.0  
high                         6.0  
                     n  sum_engaged  sum_sessions  weighted_rate
visibility_tier                                                 
low              22602        881.0       39568.0   0.0222654670
mid              22035       3136.0      113417.0   0.0276501759
high             45600      25534.0     1146823.0   0.0222649877


/tmp/ipykernel_4171/2292379323.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  signal1_weighted = sig_valid.groupby("visibility_tier", observed=True).apply(


In [17]:
# This cell is for CODE (numbers, a query, a check).

sig_valid["volume_tier"] = pd.cut(
    sig_valid["sessions_month"],
    bins=[0, 1, 9, sig_valid["sessions_month"].max()],
    labels=["1_session", "2_to_9", "10_plus"],
)

signal2_weighted = sig_valid.groupby("volume_tier", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "weighted_engagement_rate": g["engaged_sessions_month"].sum() / g["sessions_month"].sum(),
        "pct_extreme_rate": ((g["engagement_rate"] <= 0.01) | (g["engagement_rate"] >= 0.99)).mean(),
    }),
    include_groups=False,
)
print(signal2_weighted)

median_rate = sig_valid["engagement_rate"].median()
candidates = sig_valid[(sig_valid["visibility_tier"] == "low") & (sig_valid["engagement_rate"] >= median_rate)]
print(f"\nLow-visibility, above-median-engagement candidates (n={len(candidates)}) by volume tier:")
print(candidates["volume_tier"].value_counts())

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

                   n  weighted_engagement_rate  pct_extreme_rate
volume_tier                                                     
1_session    33105.0                  0.021084          1.000000
2_to_9       35925.0                  0.031929          0.894447
10_plus      21207.0                  0.021679          0.592116

Low-visibility, above-median-engagement candidates (n=22602) by volume tier:
volume_tier
1_session    15155
2_to_9        7227
10_plus        220
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
# This cell is for CODE (numbers, a query, a check).

import numpy as np
import os

VOLUME_FLOOR = 10
VISIBILITY_CUTOFF = 0.10

sig_valid["passes_gate"] = (
    (sig_valid["sessions_month"] >= VOLUME_FLOOR)
    & (sig_valid["gsc_availability_rate"] <= VISIBILITY_CUTOFF)
)
print(f"Pages passing gate: {sig_valid['passes_gate'].sum()} of {len(sig_valid)}")

sig_valid["score"] = np.where(sig_valid["passes_gate"], sig_valid["engagement_rate"], np.nan)
sig_valid["reason_code"] = np.where(sig_valid["passes_gate"], "low_visibility_high_engagement", None)
sig_valid["action"] = np.where(sig_valid["passes_gate"], "improve", "no_action")

queue = sig_valid.sort_values(["score", "sessions_month"], ascending=[False, False])

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["client_hash_id", "content_hash_id", "score", "reason_code", "action",
            "engagement_rate", "sessions_month", "gsc_availability_rate"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"\nWrote {len(queue)} rows.")
print(queue[out_cols].head(20))
print(queue[queue["passes_gate"]].head(20)["client_hash_id"].value_counts())

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Pages passing gate: 220 of 90237

Wrote 90237 rows.
                 client_hash_id           content_hash_id     score  \
210127  client_ba65e80a1116ae41  content_ea917beb21249ae7  0.368421   
43383   client_ba65e80a1116ae41  content_874f6a6f972bd217  0.307692   
43404   client_ba65e80a1116ae41  content_7854625b2ec74825  0.300000   
112951  client_3f0ce4d44fe94f3d  content_cf9fd90dd16f1aab  0.285714   
30669   client_ba65e80a1116ae41  content_dd512edf4a8d5bea  0.272727   
50807   client_cd12bcfd98942aa1  content_b7d37153f34a12e0  0.250000   
100545  client_ba65e80a1116ae41  content_dc8427b086129de9  0.250000   
50909   client_cd12bcfd98942aa1  content_33d976a3240f6b5c  0.250000   
325974  client_2094c6eb080311d5  content_d216029b156ae586  0.235294   
43428   client_ba65e80a1116ae41  content_8de42b9380d79377  0.230769   
50781   client_cd12bcfd98942aa1  content_9e3f333055ad751d  0.230769   
197082  client_ba65e80a1116ae41  content_499e767b8178062e  0.227273   
7291    client_def0955f7a

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
# This cell is for CODE (numbers, a query, a check).

# Contents with score > 0.20000 such as `content_ea917beb21249ae7`
# High priority, moderate confidence. 19 sessions and 36.8% engagement is a strong signal relative to the other candidates.
# If the 36.8% engagement is caused by tracking/session artifacts, bots, or an unusually small/non-representative sample rather than user interest.

# Contents with score ≤ 0.20000 such as `content_1c363f4eebf801d3`
# Moderate confidence. 10 sessions is the minimum volume represented here, so the 20% rate should be treated as directional.
# If the next batch of traffic produces materially lower engagement; one or two sessions can move this rate substantially.

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [22]:
# This cell is for CODE (numbers, a query, a check).

# 1. Product-flags check: FlyRank's live-app decision flags (health_score, needs_ctr_fix,
#    is_quick_win, priority_score, action_type) are excluded from this dataset by design
#    (see DATA_USE.md) -- verify that's actually true for what we pulled, not just assumed.
forbidden_flags = {"health_score", "needs_ctr_fix", "is_quick_win", "priority_score", "action_type"}
used_columns = set(sig_valid.columns) | {
    "gsc_clicks", "gsc_impressions", "ga4_engaged_sessions",
    "ga4_sessions", "ga4_total_engagement_sec", "gsc_data_available",
}
overlap = forbidden_flags & used_columns
assert not overlap, f"Product-flag leakage detected: {overlap}"
print(f"No product-flag leakage. Checked against: {sorted(forbidden_flags)}")

# 2. Future-window check: single closed month, no forward-looking data, no join
#    against fact_content_query_90d (whose *_90d / *_last30 columns overlap recent
#    months and would leak a label period -- see data-dictionary.md leakage watch).
assert MONTH == "2026-03", "Unexpected month partition"
print(f"Single-month partition: {MONTH}. No fact_content_query_90d join.")

# 3. Label-derived-input check: Lane 3 has no ground-truth label yet (unsupervised),
#    so this is trivially satisfied -- stating it explicitly rather than assuming it.
score_inputs = {"engagement_rate", "sessions_month", "gsc_availability_rate"}
print(f"Score built only from: {sorted(score_inputs)} -- raw March counts, no cluster label, no future data.")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


No product-flag leakage. Checked against: ['action_type', 'health_score', 'is_quick_win', 'needs_ctr_fix', 'priority_score']
Single-month partition: 2026-03. No fact_content_query_90d join.
Score built only from: ['engagement_rate', 'gsc_availability_rate', 'sessions_month'] -- raw March counts, no cluster label, no future data.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.